 My Cross Track project as a Data Engineer
#Nkwen Traders Data Cleaning

 Objective

 The objective of this notebook is to clean the raw Nkwen Traders sales dataset and prepare a structured product catalogue in JSON format for use by the website's Catalog builder.

 The cleaning process includes:
 
 - understanding the raw data
 - Inspecting the raw dataset
 - Handling missing values
 - Checking for duplicate transactions
 - Validating numerical values
 - Preparing the product catalogue
 - Exporting the final data as products.json

In [2]:
# Importing libraries
import pandas as pd
import json

 # Loading the CSV file
df = pd.read_csv(r"C:\Users\HP\Downloads\nkwen_traders_sales.csv")
df.head(10)


,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType
0,NKW-0001,07/02/2026,Rice 50kg,Grains,1.0,34928.0,34928,Bank Transfer,Divine K.,Walk-in
1,NKW-0002,13/04/2026,Onions 1kg,Produce,12.0,685.0,8220,Mobile Money,Florence A.,Wholesale
2,NKW-0003,11/04/2026,Salt 1kg,Groceries,11.0,297.0,3267,Orange Money,Beatrice T.,Walk-in
3,NKW-0004,01/01/2026,Bread (loaf),Bakery,13.0,591.0,7683,Bank Transfer,Divine K.,Wholesale
4,NKW-0005,14/02/2026,Beans (White),Grain,6.0,915.0,5490,Mobile Money,Beatrice T.,Walk-in
5,NKW-0006,09/03/2026,Rice 25kg,Grains,NaN,17880.0,35760,Bank Transfer,Florence A.,Walk-in
6,NKW-0007,14/01/2026,Bread (loaf),Bakery,19.0,614.0,11666,Orange Money,Florence A.,Walk-in
7,NKW-0008,11/03/2026,Rice 50kg,Grains,4.0,37414.0,149656,Bank Transfer,Ernest M.,Wholesale
8,NKW-0009,26/06/2026,Palm Oil 1L,Oils,15.0,1394.0,20910,Cash,Divine K.,Wholesale
9,NKW-0010,05/01/2026,Tomatoes 1kg,Produce,8.0,781.0,6248,Cash,Florence A.,Walk-in


From the sample data displayed above, the CSV is a sales transaction dataset, not a ready-made product catalogue. Therefore, for products.json, I would need to create one record per unique product.

In [3]:
# Inspecting the data set
df.shape     
# Tells the number of columns and rows in the data set

(500, 10)

In [4]:
df.info() 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TransactionID   500 non-null    object 
 1   Date            500 non-null    object 
 2   Product         500 non-null    object 
 3   Category        500 non-null    object 
 4   Quantity        492 non-null    float64
 5   UnitPrice_FCFA  495 non-null    float64
 6   TotalSale_FCFA  500 non-null    int64  
 7   PaymentMethod   490 non-null    object 
 8   SalesRep        494 non-null    object 
 9   CustomerType    500 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB


In [5]:
# Checking the missing values
print(df.isnull().sum())

print()
print("Total number of missing values = ",df.isnull().sum().sum())

TransactionID      0
Date               0
Product            0
Category           0
Quantity           8
UnitPrice_FCFA     5
TotalSale_FCFA     0
PaymentMethod     10
SalesRep           6
CustomerType       0
dtype: int64

Total number of missing values =  29


From above, we can see that there is a total of 29 missing values and we need to correct them before analysis.

In [6]:
# Checking for duplicate transactions
df.duplicated().sum()

np.int64(0)

In [7]:
# Check for Categories and its counts
df["Category"].value_counts()

Category
Produce       95
Groceries     82
Grains        80
Oils          74
Household     50
Groceres      30
Dairy         27
Grain         25
Bakery        22
House hold    15
Name: count, dtype: int64

In [8]:
# Fixing the categories
category_mapping = {
    "Groceres": "Groceries",
    "Grain": "Grains",
    "House hold": "Household"
}
df["Category"] = df["Category"].replace(category_mapping)
df["Category"].value_counts()

Category
Groceries    112
Grains       105
Produce       95
Oils          74
Household     65
Dairy         27
Bakery        22
Name: count, dtype: int64

In [10]:
# This command would help us identify all the Rows where quantity has a missing value.
df[df["Quantity"].isnull()]

,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType
5,NKW-0006,09/03/2026,Rice 25kg,Grains,NaN,17880.0,35760,Bank Transfer,Florence A.,Walk-in
25,NKW-0026,21/05/2026,Maggi Cubes (pack),Groceries,NaN,483.0,9177,Bank Transfer,Divine K.,Walk-in
29,NKW-0030,12/02/2026,Cassava (bag),Produce,NaN,3950.0,63200,Mobile Money,Divine K.,Wholesale
187,NKW-0188,08/04/2026,Onions 1kg,Produce,NaN,666.0,10656,Orange Money,Divine K.,Wholesale
189,NKW-0190,22/05/2026,Sugar 1kg,Groceries,NaN,728.0,2184,Bank Transfer,Ernest M.,Wholesale
353,NKW-0354,17/04/2026,Plantain (bunch),Produce,NaN,2603.0,36442,NaN,Florence A.,Walk-in
392,NKW-0393,08/04/2026,Tomato Paste (tin),Groceries,NaN,377.0,4901,Mobile Money,Ernest M.,Walk-in
444,NKW-0445,12/06/2026,Detergent 1kg,Household,NaN,1804.0,5412,Orange Money,Divine K.,Wholesale


In [11]:
# Calculating the missing Quantity values by dividing its total sale by its unit price
df["Quantity_Calculated"] = (
    df["TotalSale_FCFA"] / df["UnitPrice_FCFA"]
)


In [12]:
df[df["Quantity"].isnull()][
    ["Product", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "Quantity_Calculated"]
]


,Product,Quantity,UnitPrice_FCFA,TotalSale_FCFA,Quantity_Calculated
5,Rice 25kg,NaN,17880.0,35760,2.0
25,Maggi Cubes (pack),NaN,483.0,9177,19.0
29,Cassava (bag),NaN,3950.0,63200,16.0
187,Onions 1kg,NaN,666.0,10656,16.0
189,Sugar 1kg,NaN,728.0,2184,3.0
353,Plantain (bunch),NaN,2603.0,36442,14.0
392,Tomato Paste (tin),NaN,377.0,4901,13.0
444,Detergent 1kg,NaN,1804.0,5412,3.0


In [13]:
df["Quantity"] = df["Quantity"].fillna(df["Quantity_Calculated"])

In [14]:
df["Quantity"].isnull().sum()

np.int64(0)

In [15]:
# Missing Unit Price
df[df["UnitPrice_FCFA"].isnull()]

,TransactionID,Date,Product,Category,Quantity,UnitPrice_FCFA,TotalSale_FCFA,PaymentMethod,SalesRep,CustomerType,Quantity_Calculated
37,NKW-0038,06/04/2026,Rice 50kg,Grains,9.0,NaN,334566,Orange Money,Achu N.,Walk-in,NaN
42,NKW-0043,21/03/2026,Beans (White),Grains,19.0,NaN,16758,Bank Transfer,Ernest M.,Walk-in,NaN
125,NKW-0126,14/06/2026,Rice 50kg,Grains,13.0,NaN,485563,Orange Money,Ernest M.,Wholesale,NaN
149,NKW-0150,05/02/2026,Bread (loaf),Bakery,3.0,NaN,1827,Orange Money,Achu N.,Walk-in,NaN
400,NKW-0401,26/03/2026,Matches (box),Household,3.0,NaN,294,Orange Money,Beatrice T.,Walk-in,NaN


From above we can see that there are 5 rows with missing values for Unit price

In [16]:
df["CalculatedUnitPrice"] = (
    df["TotalSale_FCFA"] / df["Quantity"]
)

In [17]:
df[df["UnitPrice_FCFA"].isnull()][
    ["Product", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "CalculatedUnitPrice"]
]

,Product,Quantity,UnitPrice_FCFA,TotalSale_FCFA,CalculatedUnitPrice
37,Rice 50kg,9.0,NaN,334566,37174.0
42,Beans (White),19.0,NaN,16758,882.0
125,Rice 50kg,13.0,NaN,485563,37351.0
149,Bread (loaf),3.0,NaN,1827,609.0
400,Matches (box),3.0,NaN,294,98.0


In [18]:
df["UnitPrice_FCFA"] = df["UnitPrice_FCFA"].fillna(df["CalculatedUnitPrice"])

In [19]:
# Checking
df["CalculatedTotal"] = (
    df["Quantity"] * df["UnitPrice_FCFA"]
)
df["SaleDifference"] = (
    df["TotalSale_FCFA"] - df["CalculatedTotal"]
)
df["SaleDifference"].abs().sum()

np.float64(0.0)

In [20]:
# Removing temporary questions
df.drop(
    columns = ["Quantity_Calculated", "CalculatedUnitPrice", "CalculatedTotal", "SaleDifference"],
    errors = "ignore",
    inplace = True
)

In [21]:
# Handle missing PaymentMethod

df["PaymentMethod"] = df["PaymentMethod"].fillna("Unknown")

In [22]:
# Handle missing SalesRep
df["SalesRep"] = df["SalesRep"].fillna("Unknown")

In [24]:
# Checking for impossible values
df["Quantity"].describe()

count    500.000000
mean      10.988000
std       10.220802
min        1.000000
25%        5.000000
50%       11.000000
75%       15.000000
max      200.000000
Name: Quantity, dtype: float64

In [25]:
# Checks
print((df["Quantity"] < 0).sum())
print((df["UnitPrice_FCFA"] < 0).sum())
print((df["TotalSale_FCFA"] < 0).sum())

0
0
0


In [26]:
# Checking for the final missing 
df.isnull().sum()

TransactionID     0
Date              0
Product           0
Category          0
Quantity          0
UnitPrice_FCFA    0
TotalSale_FCFA    0
PaymentMethod     0
SalesRep          0
CustomerType      0
dtype: int64

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TransactionID   500 non-null    object 
 1   Date            500 non-null    object 
 2   Product         500 non-null    object 
 3   Category        500 non-null    object 
 4   Quantity        500 non-null    float64
 5   UnitPrice_FCFA  500 non-null    float64
 6   TotalSale_FCFA  500 non-null    int64  
 7   PaymentMethod   500 non-null    object 
 8   SalesRep        500 non-null    object 
 9   CustomerType    500 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 39.2+ KB


In [28]:
df["Product"].nunique()


20

In [29]:
df["Product"].unique()

array(['Rice 50kg', 'Onions 1kg', 'Salt 1kg', 'Bread (loaf)',
       'Beans (White)', 'Rice 25kg', 'Palm Oil 1L', 'Tomatoes 1kg',
       'Palm Oil 5L', 'Tomato Paste (tin)', 'Detergent 1kg',
       'Beans (Red)', 'Matches (box)', 'Cassava (bag)', 'Sugar 1kg',
       'Maggi Cubes (pack)', 'Vegetable Oil 5L', 'Plantain (bunch)',
       'Milk Powder 400g', 'Soap (bar)'], dtype=object)

In [30]:
# Creating the product catalog
df_sorted = df.sort_values("Date")
products = (
    df_sorted.drop_duplicates("Product", keep = "last")   # Keeps the details of only the last in the sorted list
    [["Product", "Category", "UnitPrice_FCFA"]].copy()
)

In [31]:
products = products.rename(columns = {
    "Product": "name",
    "Category": "category",
    "UnitPrice_FCFA": "price"
})
products

,name,category,price
46,Tomato Paste (tin),Groceries,348.0
96,Onions 1kg,Produce,687.0
179,Rice 25kg,Grains,17935.0
286,Soap (bar),Household,398.0
496,Palm Oil 5L,Oils,7010.0
32,Palm Oil 1L,Oils,1400.0
117,Maggi Cubes (pack),Groceries,478.0
355,Beans (Red),Grains,890.0
301,Rice 50kg,Grains,36222.0
172,Beans (White),Grains,818.0


In [32]:
# Creating the notebook
products.to_json( "products.json", orient = "records", indent = 4)

In [33]:
# Check in json
with open ("products.json", "r", encoding = "utf-8") as file:
    product_data = json.load(file)
product_data[:3]    

[{'name': 'Tomato Paste (tin)', 'category': 'Groceries', 'price': 348.0},
 {'name': 'Onions 1kg', 'category': 'Produce', 'price': 687.0},
 {'name': 'Rice 25kg', 'category': 'Grains', 'price': 17935.0}]

Conclusion
 The raw Nkwen Traders sales dataset was inspected and cleaned before being prepared for use by the website. here are the steps i followed:

 The cleaning process includes:
 - Checking the dataset structure and data type.
 - Identifying and handling missing values.
 - Standardizing inconsistent category names.
 - Correcting the "Mobile Money" payment-method typo
 - Checking for duplicate transactions
 - Converting dates to a proper datetime format
 - Validating the relationship between: quantity, unit price, and total sale
 - Creating a unique product price
 - Exporting the catalog as a json fine
